In [1]:
%reload_ext autoreload
%autoreload 2
%cd ../../

/home/hazzu/Code/thesis


In [2]:
import os
import os.path as osp
from pathlib import Path
import shutil
from tqdm import tqdm

SOURCE_DIR = "datasets/ver2/selfies_1"
TARGET_DIR = "datasets/ver2/image_folder"

image_idx = 0
os.makedirs(TARGET_DIR, exist_ok=True)

for idx, folder_name in tqdm(enumerate(os.listdir(SOURCE_DIR))):
    folder_path = osp.join(SOURCE_DIR, folder_name)
    if not osp.isdir(folder_path):
        continue

    target_folder_name = "0_0_" + ("0" * (7 - len(str(idx))) + str(idx))
    target_folder_path = osp.join(TARGET_DIR, target_folder_name)
    os.makedirs(target_folder_path, exist_ok=True)

    for image_name in os.listdir(folder_path):
        extension = Path(image_name).suffix

        src_path = osp.join(folder_path, image_name)
        dst_path = osp.join(
            target_folder_path,
            "0_" + str(image_idx) + extension,
        )

        shutil.copyfile(src_path, dst_path)
        image_idx += 1

98it [00:00, 385.11it/s]


In [3]:
import cv2, os
from tqdm import tqdm
from lib.face_detector.retinaface import RetinaFaceDetector
from lib.face_recognizer.arcface import ArcFaceRecognizer
from insightface.utils import face_align

retinaface = RetinaFaceDetector()
arcface = ArcFaceRecognizer()

for folder_name in tqdm(os.listdir(TARGET_DIR)):
    folder_path = osp.join(TARGET_DIR, folder_name)

    for image_name in os.listdir(folder_path):
        image_path = osp.join(folder_path, image_name)
        image = cv2.imread(image_path)

        faces = retinaface.detect(image)
        if len(faces) == 0:
            print(f"Face not found in {image_path}")
            continue
        if len(faces) > 1:
            print(f"{len(faces)} faces found in {image_path}")
            continue

        face = faces[0]
        arcface_face = arcface._convert_input_face(face)
        aligned_face = face_align.norm_crop(
            image,
            landmark=arcface_face.kps,
            image_size=arcface._recognizer.input_size[0],
        )

        cv2.imwrite(image_path, aligned_face)

/home/hazzu/Code/thesis/.conda/lib/python3.12/site-packages/albumentations/__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.8' (you have '2.0.5'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
100%|██████████| 98/98 [00:56<00:00,  1.74it/s]


In [4]:
!python3 im2rec.py train datasets/ver2/image_folder --recursive --list

0_0_0000000 0
0_0_0000001 1
0_0_0000002 2
0_0_0000003 3
0_0_0000004 4
0_0_0000005 5
0_0_0000006 6
0_0_0000007 7
0_0_0000008 8
0_0_0000009 9
0_0_0000010 10
0_0_0000011 11
0_0_0000012 12
0_0_0000013 13
0_0_0000014 14
0_0_0000015 15
0_0_0000016 16
0_0_0000017 17
0_0_0000018 18
0_0_0000019 19
0_0_0000020 20
0_0_0000021 21
0_0_0000022 22
0_0_0000023 23
0_0_0000024 24
0_0_0000025 25
0_0_0000026 26
0_0_0000027 27
0_0_0000028 28
0_0_0000029 29
0_0_0000030 30
0_0_0000031 31
0_0_0000032 32
0_0_0000033 33
0_0_0000034 34
0_0_0000035 35
0_0_0000036 36
0_0_0000037 37
0_0_0000038 38
0_0_0000039 39
0_0_0000040 40
0_0_0000041 41
0_0_0000042 42
0_0_0000043 43
0_0_0000044 44
0_0_0000045 45
0_0_0000046 46
0_0_0000047 47
0_0_0000048 48
0_0_0000049 49
0_0_0000050 50
0_0_0000051 51
0_0_0000052 52
0_0_0000053 53
0_0_0000054 54
0_0_0000055 55
0_0_0000056 56
0_0_0000057 57
0_0_0000058 58
0_0_0000059 59
0_0_0000060 60
0_0_0000061 61
0_0_0000062 62
0_0_0000063 63
0_0_0000064 64
0_0_0000065 65
0_0_0000066 66
0_0_0

In [5]:
!python3 im2rec.py train datasets/ver2/image_folder --num-thread 16 --quality 100

Creating .rec file from /home/hazzu/Code/thesis/train.lst in /home/hazzu/Code/thesis
time: 0.003977775573730469  count: 0
